# Basic Solver Demo

This notebook turns the tiny 2D LP into a fully documented experiment. It states the optimization problem explicitly, runs both the adaptive solver and the SciPy/HiGHS baseline, and inspects the optimization trace so the gap between a good approximation and a certified solution is visible.


In [1]:
from __future__ import annotations

import numpy as np
from IPython.display import Markdown, display

np.set_printoptions(suppress=True, precision=6)


def fmt(value, digits: int = 6) -> str:
    if value is None:
        return 'None'
    if isinstance(value, (np.bool_, bool)):
        return 'True' if bool(value) else 'False'
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)):
        return f'{float(value):.{digits}f}'
    if isinstance(value, str):
        return value
    if isinstance(value, tuple):
        return '[' + ', '.join(fmt(v, digits=digits) for v in value) + ']'
    if isinstance(value, (list, np.ndarray)):
        arr = np.asarray(value)
        if arr.ndim == 0:
            return fmt(arr.item(), digits=digits)
        return '[' + ', '.join(fmt(v, digits=digits) for v in arr.tolist()) + ']'
    return str(value)


def markdown_table(headers, rows):
    lines = [
        '| ' + ' | '.join(headers) + ' |',
        '| ' + ' | '.join(['---'] * len(headers)) + ' |',
    ]
    for row in rows:
        lines.append('| ' + ' | '.join(str(cell) for cell in row) + ' |')
    return '\n'.join(lines)


def show_table(headers, rows, title: str | None = None):
    parts = []
    if title:
        parts.append(f'### {title}')
    parts.append(markdown_table(headers, rows))
    display(Markdown('\n\n'.join(parts)))


def show_bullets(title: str, items):
    display(Markdown('### ' + title + '\n' + '\n'.join(f'- {item}' for item in items)))


def mask_to_bits(mask) -> str:
    return ''.join('1' if bool(v) else '0' for v in mask)

from time import perf_counter

from lpas.experiments.generators import tiny_known_lp
from lpas.solver.adaptive_solver import AdaptiveLPSolver
from lpas.solver.scipy_handoff import compare_adaptive_to_scipy, solve_with_scipy
from lpas.utils.config import SamplerConfig, SolverConfig


## Problem Setup

The package example uses a dense maximization LP in the standard repository form `Ax <= b`, `x >= 0`. Because the problem is two-dimensional, it is easy to interpret geometrically: the optimal point is expected to live at a vertex where multiple constraints meet.


In [2]:
problem = tiny_known_lp()

constraint_rows = [
    ('c1', '`x1 + x2 <= 4`', 'shared capacity constraint'),
    ('c2', '`x1 <= 2`', 'upper bound on `x1`'),
    ('c3', '`x2 <= 3`', 'upper bound on `x2`'),
]
show_table(['Constraint', 'Expression', 'Interpretation'], constraint_rows, title='Primal LP')
display(Markdown('**Objective:** maximize `3 x1 + 2 x2` with the implicit nonnegativity constraints `x1 >= 0`, `x2 >= 0`.'))
display(Markdown(f'`A = {fmt(problem.A)}`, `b = {fmt(problem.b)}`, `c = {fmt(problem.c)}`'))


### Primal LP

| Constraint | Expression | Interpretation |
| --- | --- | --- |
| c1 | `x1 + x2 <= 4` | shared capacity constraint |
| c2 | `x1 <= 2` | upper bound on `x1` |
| c3 | `x2 <= 3` | upper bound on `x2` |

**Objective:** maximize `3 x1 + 2 x2` with the implicit nonnegativity constraints `x1 >= 0`, `x2 >= 0`.

`A = [[1.000000, 1.000000], [1.000000, 0.000000], [0.000000, 1.000000]]`, `b = [4.000000, 2.000000, 3.000000]`, `c = [3.000000, 2.000000]`

## Reproducible Configuration

I keep the same configuration style as the example scripts: moderate batch size, a fixed seed, and enough patience to let the Gaussian sampler settle without making the notebook slow to rerun.


In [3]:
config = SolverConfig(
    batch_size=512,
    max_iter=80,
    patience=20,
    sampler=SamplerConfig(
        seed=42,
        sigma_init=2.5,
        primal_init_mean=0.75,
        dual_init_mean=0.75,
    ),
)
config


SolverConfig(batch_size=512, max_iter=80, elite_fraction=0.1, seed=0, active_tol=1e-06, feasibility_tol=1e-07, gap_tol=1e-06, patience=20, time_limit_seconds=None, archive_limit_multiplier=5, variance_collapse_factor=1.0001, scoring=ScoringConfig(w_primal_obj=1.0, w_dual_obj=0.25, w_gap=1.5, w_pviol=2.0, w_dviol=2.0, w_comp=1.0, w_geo=0.2, w_active=0.2, geometry_sigma=1.0, geometry_dual_weight=0.5, active_similarity_beta=0.5, cluster_smoothing=0.0), sampler=SamplerConfig(seed=42, alpha=0.7, sigma_init=2.5, sigma_min=1e-06, sigma_max=10.0, primal_init_mean=0.75, dual_init_mean=0.75, max_retries=8, primal_violation_threshold=1.0, dual_violation_threshold=1.0), warm_start=WarmStartConfig(feasibility_tol=1e-07, max_combinations=512), vertex_polishing=VertexPolishingConfig(enabled=True, elite_fraction=0.05, tau=0.01, method='rbf', max_ranked_constraints=None, max_candidates_per_sample=100, max_total_candidates=5000, feasibility_tol=1e-08, residual_tol=1e-08))

## Adaptive Run Versus SciPy/HiGHS

The adaptive solver is the research prototype under study. SciPy/HiGHS is the reference baseline used to measure objective quality and recover the exact vertex.


In [4]:
adaptive_start = perf_counter()
adaptive = AdaptiveLPSolver(config).solve(problem)
adaptive_seconds = perf_counter() - adaptive_start

scipy_start = perf_counter()
scipy = solve_with_scipy(problem)
scipy_seconds = perf_counter() - scipy_start

comparison = compare_adaptive_to_scipy(adaptive, scipy)


In [5]:
summary_rows = [
    (
        'Adaptive sampler',
        adaptive.status.value,
        fmt(adaptive.best_primal_objective),
        fmt(adaptive.best_x),
        fmt(adaptive.best_gap),
        fmt(adaptive.best_primal_violation),
        fmt(adaptive.best_dual_violation),
        adaptive.iterations,
        f'{adaptive_seconds:.3f}',
    ),
    (
        'SciPy / HiGHS',
        'SUCCESS' if scipy.success else f'FAIL ({scipy.status})',
        fmt(scipy.objective),
        fmt(scipy.x),
        'n/a',
        'n/a',
        'n/a',
        'n/a',
        f'{scipy_seconds:.3f}',
    ),
]
show_table(
    ['Method', 'Status', 'Objective', 'Candidate x', 'Raw gap', 'Primal viol.', 'Dual viol.', 'Iterations', 'Time (s)'],
    summary_rows,
    title='Single-problem comparison',
)
relative_error_pct = 100.0 * comparison.relative_objective_error if comparison.relative_objective_error is not None else None
display(Markdown(f'Adaptive relative objective error versus SciPy: `{relative_error_pct:.3f}%`.'))


### Single-problem comparison

| Method | Status | Objective | Candidate x | Raw gap | Primal viol. | Dual viol. | Iterations | Time (s) |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Adaptive sampler | APPROXIMATE | 10.000000 | [2.000000, 2.000000] | 0.012261 | 0.000000 | 0.000000 | 80 | 3.687 |
| SciPy / HiGHS | SUCCESS | 10.000000 | [2.000000, 2.000000] | n/a | n/a | n/a | n/a | 0.007 |

Adaptive relative objective error versus SciPy: `0.000%`.

## Optimization Trace

The history object records what the solver knew at each iteration. The important fields for this notebook are the best feasible objective found so far, whether any certified gap has appeared, and how quickly the average violations collapse.


In [6]:
checkpoint_indices = sorted(set([0, 9, 19, 39, 59, len(adaptive.history) - 1]))
history_rows = []
for index in checkpoint_indices:
    entry = adaptive.history[index]
    history_rows.append(
        (
            entry.iteration,
            fmt(entry.best_score),
            fmt(entry.best_feasible_primal_objective),
            fmt(entry.best_certified_gap),
            fmt(entry.mean_primal_violation),
            fmt(entry.mean_dual_violation),
        )
    )
show_table(
    ['Iter.', 'Best score', 'Best feasible obj.', 'Best certified gap', 'Mean primal viol.', 'Mean dual viol.'],
    history_rows,
    title='Checkpoints from the adaptive run',
)


### Checkpoints from the adaptive run

| Iter. | Best score | Best feasible obj. | Best certified gap | Mean primal viol. | Mean dual viol. |
| --- | --- | --- | --- | --- | --- |
| 0 | 6.679159 | 9.668214 | 1.107172 | 1.209652 | 1.695539 |
| 9 | 6.668787 | 9.921708 | 0.249709 | 0.175730 | 0.677729 |
| 19 | 6.615362 | 9.990371 | 0.158833 | 0.041622 | 0.151422 |
| 39 | 7.136301 | 9.990371 | 0.045634 | 0.007129 | 0.013444 |
| 59 | 7.124070 | 9.990371 | 0.032454 | 0.001270 | 0.002281 |
| 79 | 6.654599 | 9.990371 | 0.021890 | 0.000203 | 0.000544 |

In [7]:
insights = [
    f'The adaptive solver reached objective `{adaptive.best_primal_objective:.6f}` versus SciPy\'s `{scipy.objective:.6f}`, so the sampler found a near-optimal point even without exact certification.',
    f'The run stopped with status `{adaptive.status.value}` because no primal-dual pair achieved the configured gap tolerance; the best recorded raw gap remained `{adaptive.best_gap:.6f}`.',
    f'Feasibility was already strong at the selected point: primal violation `{adaptive.best_primal_violation:.6f}` and dual violation `{adaptive.best_dual_violation:.6f}`.',
    f'The warm-start stage reported `{adaptive.warm_start_hint.message}`, which hints that active-set recovery is the limiting step rather than gross objective error.',
]
show_bullets('Insights', insights)


### Insights
- The adaptive solver reached objective `10.000000` versus SciPy's `10.000000`, so the sampler found a near-optimal point even without exact certification.
- The run stopped with status `APPROXIMATE` because no primal-dual pair achieved the configured gap tolerance; the best recorded raw gap remained `0.012261`.
- Feasibility was already strong at the selected point: primal violation `0.000000` and dual violation `0.000000`.
- The warm-start stage reported `Feasible vertex reconstructed from soft active-set polishing`, which hints that active-set recovery is the limiting step rather than gross objective error.